# Controllable Multi-Axis Music Search — Colab Training

Runs the ingest + head-training pipeline from `music-similarity-mle-plan.md`
on Google Colab instead of the Cornell cluster it was originally scoped for.

**Why this notebook looks different from a typical training notebook:** Colab's
GPU runtimes only have ~64GB of *ephemeral* local disk (wiped every time the
runtime recycles), and the full ingest plan calls for separating 25-30k tracks
into 4 stems each — that's several hundred GB of uncompressed WAV if stored
naively. So this pipeline never holds more than one track's stems on disk at a
time: separate → embed → delete, immediately, with only the small embedding
vectors persisted to Google Drive. See `pipeline/colab_pipeline.py` for the
full rationale.

**Before you start:** Runtime → Change runtime type → GPU (T4 to start; try L4
if it's offered and your budget allows). Then run the cells top to bottom —
every long-running cell is checkpointed to Drive, so if Colab disconnects you
just re-run the same cell and it picks up where it left off.

## 1. Environment check + install

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU -- go to Runtime > Change runtime type > GPU before continuing.")


In [ ]:
# Colab already ships a CUDA-matched torch/torchaudio; requirements-colab.txt
# deliberately excludes them so this install doesn't clobber that wiring.
!pip install -q -r requirements-colab.txt


## 2. Mount Google Drive

Do this **before** adding anything under `/content/drive` to `sys.path` --
the path doesn't exist until this cell finishes, so importing project code
from Drive earlier than this will fail with `ModuleNotFoundError`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. Point Python at the project code

Edit `PROJECT_DIR` below to match exactly where you put the `music-similarity`
folder in your Drive (case-sensitive; it's `MyDrive`, not `My Drive`).

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/music-similarity")  # <-- edit if yours differs
assert PROJECT_DIR.exists(), f"{PROJECT_DIR} not found -- check the path and that Drive finished mounting"
sys.path.insert(0, str(PROJECT_DIR))


## 4. Point the pipeline at persistent storage

In [ ]:
from pipeline.config import configure_for_colab
from pipeline.colab_pipeline import gpu_report

# Repoints pipeline.config.PATHS / QDRANT at <PROJECT_DIR>/colab_run so
# embeddings, checkpoints, and the local Qdrant index all persist across
# runtime recycles instead of living on ephemeral local disk.
drive_root = PROJECT_DIR / "colab_run"
configure_for_colab(drive_root)
print(f"Data root: {drive_root}")
print(gpu_report())


## 5. Size your subset to your compute budget

Colab bills GPU time in compute units (CU); rates as of mid-2026 are roughly
1.76 CU/hr on a T4 and 15 CU/hr on an A100, with L4 in between. This cell
gives a rough sizing estimate — treat it as a planning number, not a
guarantee, since actual per-track time depends on track length and whatever
else is running on the shared GPU.

In [ ]:
from pipeline.colab_pipeline import detect_gpu, estimate_hours, estimate_feasible_tracks

COMPUTE_UNITS_BUDGET = 100  # set to what you actually have

gpu = detect_gpu()
print("Detected GPU:", gpu)
hours = estimate_hours(COMPUTE_UNITS_BUDGET, gpu) if gpu else None
tracks = estimate_feasible_tracks(COMPUTE_UNITS_BUDGET, gpu) if gpu else None
if hours:
    print(f"~{hours:.1f} GPU-hours available on this GPU for {COMPUTE_UNITS_BUDGET} CU")
    print(f"~{tracks} tracks feasible through Demucs+MERT (leaves headroom for Phase 2 training + retries)")
else:
    print("Unrecognized/no GPU -- pick a subset size conservatively (a few thousand tracks) and watch actual timing on the first batch instead.")


**Recommendation for a 100 CU budget:** on a T4 (~57 hrs total), a
5,000-8,000 track subset for ingest leaves comfortable headroom for Phase 2
head training and mistakes/retries, while still being large enough for
meaningful precision@k eval. Scale down further if you're sharing the GPU
with other Colab usage this session.

## 6. Phase 0 — walking skeleton (whole-mix MERT + FAISS)

De-risks the rest of the notebook on a tiny sample before committing GPU-hours to the full ingest.

In [ ]:
from pipeline.download_jamendo import fetch_metadata, read_track_ids, stratified_subset, download_audio
from pipeline.config import PATHS

metadata_path = fetch_metadata("autotagging_top50tags", PATHS.metadata_dir / "jamendo.tsv")
all_rows = read_track_ids(metadata_path)
baseline_rows = stratified_subset(all_rows, n=200, seed=0)
download_audio(baseline_rows, PATHS.audio_dir)
print(f"Downloaded {len(baseline_rows)} tracks for the baseline skeleton")


In [ ]:
from pipeline.baseline_faiss import build_index, query_index
from pipeline.config import PATHS

baseline_index_path = PATHS.embeddings_dir / "baseline.index"
build_index(PATHS.audio_dir, baseline_index_path)

# quick sanity listen -- pick any downloaded track id
sample_track_id = baseline_rows[0]["TRACK_ID"].replace("track_", "")
for track_id, score in query_index(baseline_index_path, sample_track_id, k=5):
    print(f"{track_id}\t{score:.3f}")


## 7. Phase 1 — streaming ingest (main event)

Downloads and processes tracks in a loop that separates → embeds → deletes
per track, appending to a Drive-backed JSONL manifest after every track. Safe
to interrupt: re-running this cell skips whatever's already in the manifest.

In [ ]:
N_TRACKS = 5000  # adjust per the budget estimate above

subset_rows = stratified_subset(all_rows, n=N_TRACKS, seed=1)
subset_dir = PATHS.audio_dir / "ingest_subset"
subset_dir.mkdir(parents=True, exist_ok=True)
download_audio(subset_rows, subset_dir)

audio_paths = sorted(subset_dir.glob("*.mp3"))
print(f"{len(audio_paths)} tracks staged for streaming ingest")


In [ ]:
from pipeline.colab_pipeline import run_streaming_ingest, check_disk_headroom
from pipeline.config import PATHS

check_disk_headroom(min_free_gb=5.0)

manifest_path = PATHS.embeddings_dir / "manifest.jsonl"  # lives under drive_root/data -> persists
run_streaming_ingest(audio_paths, manifest_path)


In [ ]:
from pipeline.colab_pipeline import manifest_to_parquet
from pipeline.config import PATHS

raw_embeddings_parquet = PATHS.embeddings_dir / "mert_embeddings.parquet"
manifest_to_parquet(manifest_path, raw_embeddings_parquet)


## 8. Phase 2 — aspect head training

Reserves a smaller sub-subset for pair (anchor/positive) ingest, since this
pass runs Demucs *and* the invariance augmentations *and* two embedding calls
per stem — roughly double the compute of Phase 1's ingest per track.

In [ ]:
N_PAIR_TRACKS = 1500  # smaller than N_TRACKS -- Phase 2 is compute-heavier per track

pair_rows = stratified_subset(all_rows, n=N_PAIR_TRACKS, seed=2)
pair_dir = PATHS.audio_dir / "pair_subset"
pair_dir.mkdir(parents=True, exist_ok=True)
download_audio(pair_rows, pair_dir)
pair_audio_paths = sorted(pair_dir.glob("*.mp3"))

from pipeline.colab_pipeline import run_streaming_pair_ingest

anchor_manifest = PATHS.embeddings_dir / "pair_anchor.jsonl"
positive_manifest = PATHS.embeddings_dir / "pair_positive.jsonl"
run_streaming_pair_ingest(pair_audio_paths, anchor_manifest, positive_manifest)

anchor_parquet = manifest_to_parquet(anchor_manifest, PATHS.embeddings_dir / "pair_anchor.parquet")
positive_parquet = manifest_to_parquet(positive_manifest, PATHS.embeddings_dir / "pair_positive.parquet")


In [ ]:
from training.train import train_one_aspect

trained_heads = {}
for aspect in ["rhythm", "melody", "timbre", "vocal"]:
    print(f"--- training {aspect} head ---")
    trained_heads[aspect] = train_one_aspect(
        aspect=aspect,
        anchor_path=anchor_parquet,
        positive_path=positive_parquet,
        loss_name="info_nce",
        epochs=15,  # short run to fit a Colab session; raise once you've validated the loop
    )


Checkpoints land in `<drive_root>/data/checkpoints/<aspect>_head.pt` — persisted, so later sessions can `torch.load` them directly instead of retraining.

## 9. Eval

Triplet accuracy needs a triplet pool; the quickest path on Colab is generating invariance-style triplets from the pair manifests you already have (real MoisesDB/MUSDB triplets are the paper track's job, run separately once those corpora are downloaded).

In [ ]:
import json
import numpy as np
import pandas as pd

from eval.precision_at_k import build_tag_index, precision_at_k

# Precision@k against Jamendo tags, using the raw (untrained) per-stem MERT
# embeddings from Phase 1 as a quick sanity baseline before trusting the
# trained heads' numbers.
df = pd.read_parquet(raw_embeddings_parquet)
drum_only = df[df["stem"] == "drums"]
embeddings = {row["track_id"]: np.asarray(row["embedding"]) for _, row in drum_only.iterrows()}
tag_index = build_tag_index(metadata_path)

p_at_10 = precision_at_k(embeddings, tag_index, k=10, n_queries=200)
print(f"Baseline (raw drum-stem MERT) precision@10: {p_at_10:.4f}")


## 10. Index into local Qdrant + demo query

Uses Qdrant's embedded local mode (`QdrantClient(path=...)`) -- no server process needed, and it's already pointed at Drive by `setup_colab` above.

In [ ]:
from training.aspect_heads import MultiAspectModel
import torch

model = MultiAspectModel()
for aspect, head in model.heads.items():
    ckpt = PATHS.checkpoints_dir / f"{aspect}_head.pt"
    if ckpt.exists():
        head.load_state_dict(torch.load(ckpt, map_location="cpu"))
        head.eval()

# Project every track in the Phase-1 raw-embeddings parquet through the
# trained heads to build the aspect-vector table qdrant_index.py expects.
rows = []
for track_id, group in df.groupby("track_id"):
    stem_embeddings = {
        row["stem"]: torch.tensor(row["embedding"], dtype=torch.float32).unsqueeze(0) for _, row in group.iterrows()
    }
    with torch.no_grad():
        projected = model.project_all(stem_embeddings)
    row = {"track_id": track_id}
    for aspect, vec in projected.items():
        row[aspect] = vec.squeeze(0).numpy().tolist()
    rows.append(row)

aspect_df = pd.DataFrame(rows)
aspect_parquet = PATHS.embeddings_dir / "aspect_vectors.parquet"
aspect_df.to_parquet(aspect_parquet)
print(f"Projected {len(aspect_df)} tracks through trained heads")


In [ ]:
from pipeline.qdrant_index import get_client, ensure_collection, load_aspect_table, upsert_tracks

client = get_client()
ensure_collection(client)
table = load_aspect_table(aspect_parquet)
upsert_tracks(client, table)
print(f"Indexed {len(table)} tracks into local Qdrant at {drive_root / 'qdrant_local'}")


In [ ]:
from api.search import weighted_fusion_search, fetch_seed_vectors

seed_id = table.iloc[0]["track_id"]
seed_vectors = fetch_seed_vectors(client, seed_id)
weights = {"rhythm": 0.4, "melody": 0.2, "timbre": 0.2, "vocal": 0.1, "lyric": 0.1}
results, latency_ms = weighted_fusion_search(client, seed_vectors, weights, top_k=10, exclude_id=seed_id)

print(f"Seed: {seed_id}  (latency {latency_ms:.1f} ms)")
for r in results:
    print(f"  {r['track_id']}\tscore={r['score']:.3f}")


## Next steps

- Bump `N_TRACKS` / `N_PAIR_TRACKS` and re-run once you've confirmed timing on
  the smaller subset — the manifests are append-only, so scaling up just
  processes the delta.
- For the paper track's real-stem triplet engine (`training/triplet_engine.py`),
  download MoisesDB/MUSDB18-HQ to Drive separately (they're small enough to
  keep around) and point it at that folder.
- To preview the FastAPI + Streamlit app from inside Colab, run `uvicorn
  api.main:app` and `streamlit run web/app.py` in background cells and tunnel
  with `pyngrok` (already in requirements-colab.txt) -- useful for a demo, not
  a substitute for deploying it properly per Phase 3 of the plan.